In [237]:
!pip -q install sentence-transformers faiss-cpu

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [238]:
import sys
import sys
!{sys.executable} -m pip install --upgrade typing_extensions beautifulsoup4
print(sys.executable)

c:\Users\krbha\AppData\Local\Programs\Python\Python39\python.exe


You should consider upgrading via the 'c:\Users\krbha\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


In [239]:
import torch
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
import faiss

print("torch:", torch.__version__)
print("All imports successful")

torch: 2.8.0+cpu
All imports successful


In [240]:
import requests
import re
import pandas as pd
import numpy as np

from bs4 import BeautifulSoup, NavigableString

from datetime import date

from sentence_transformers import SentenceTransformer

import faiss
import pickle

from IPython.display import display

In [241]:
sources = {
    "EXT-01": {
        "category": "Extension",
        "title": "Extensions",
        "url": "https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/extensions",
        "type": "webpage"
    },

    "SC-01": {
        "category": "Special Consideration",
        "title": "Special Consideration",
        "url": "https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/special-consideration",
        "type": "webpage"
    },

    "EAA-01": {
        "category": "Equitable Assessment Arrangements",
        "title": "Equitable Assessment Arrangements",
        "url": "https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/equitable-assessment-arrangements",
        "type": "webpage"
    },

    "EAA-02": {
        "category": "Equitable Assessment Arrangements",
        "title": "Equitable Learning FAQs",
        "url": "https://www.rmit.edu.au/students/support-services/equitable-learning/faqs",
        "type": "webpage"
    },

    "POL-01": {
        "category": "Policy",
        "title": "Assessment and Assessment Flexibility Policy",
        "url": "https://policies.rmit.edu.au/document/view.php?id=7",
        "type": "policy"
    }
}

print("Number of sources:", len(sources))

for source_id, source in sources.items():
    print(
        source_id,
        "→",
        source["title"]
    )

Number of sources: 5
EXT-01 → Extensions
SC-01 → Special Consideration
EAA-01 → Equitable Assessment Arrangements
EAA-02 → Equitable Learning FAQs
POL-01 → Assessment and Assessment Flexibility Policy


In [242]:
def get_webpage(url):

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/139.0 Safari/537.36"
        ),
        "Accept-Language": "en-AU,en;q=0.9"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    return response

In [243]:
response = get_webpage(
    sources["EXT-01"]["url"]
)

print("Status code:", response.status_code)
print("Downloaded characters:", len(response.text))

Status code: 200
Downloaded characters: 174148


In [244]:
def scrape_rmit_policy(source_id, source):

    response = get_webpage(source["url"])

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # Remove irrelevant page elements
    for tag in soup.find_all([
        "script",
        "style",
        "noscript",
        "nav",
        "footer",
        "header",
        "iframe",
        "form",
        "aside"
    ]):
        tag.decompose()

    records = []

    # --------------------------------------------------
    # Find policy headings
    # --------------------------------------------------

    headings = soup.find_all(
        ["h1", "h2", "h3", "h4"]
    )

    print(
        "Policy headings found:",
        len(headings)
    )

    for heading in headings:

        section_title = clean_text(
            heading.get_text(
                " ",
                strip=True
            )
        )

        if not section_title:
            continue

        content_parts = []

        # --------------------------------------------------
        # Walk through following elements
        # --------------------------------------------------

        for element in heading.find_all_next():

            # Stop at next heading of same/higher level
            if element.name in [
                "h1",
                "h2",
                "h3",
                "h4"
            ]:

                if element != heading:

                    # For policy sections, stop at any
                    # subsequent heading.
                    break

            # Paragraphs
            if element.name == "p":

                text = clean_text(
                    element.get_text(
                        " ",
                        strip=True
                    )
                )

                if text:
                    content_parts.append(text)

            # List items
            elif element.name == "li":

                text = clean_text(
                    element.get_text(
                        " ",
                        strip=True
                    )
                )

                if text:
                    content_parts.append(text)

            # Table cells
            elif element.name in [
                "td",
                "th"
            ]:

                text = clean_text(
                    element.get_text(
                        " ",
                        strip=True
                    )
                )

                if text:
                    content_parts.append(text)

        # --------------------------------------------------
        # Combine content
        # --------------------------------------------------

        content = " ".join(
            content_parts
        )

        content = clean_text(
            content
        )

        # Ignore tiny sections
        if len(content) < 50:
            continue

        records.append({

            "source_id":
                source_id,

            "category":
                source["category"],

            "title":
                source["title"],

            "section":
                section_title,

            "content":
                content,

            "source_url":
                source["url"],

            "source_type":
                source["type"],

            "collection_date":
                str(date.today()),

            "effective_date":
                "2026-07-28"

        })

    return records

In [245]:
def clean_text(text):

    text = str(text)

    # Replace non-breaking spaces
    text = text.replace("\xa0", " ")

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [246]:
def scrape_rmit_page(source_id, source):

    response = get_webpage(source["url"])

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # --------------------------------------------------
    # Remove elements that are never useful for the RAG
    # --------------------------------------------------

    for tag in soup.find_all([
        "script",
        "style",
        "noscript",
        "nav",
        "footer",
        "header",
        "iframe",
        "form"
    ]):
        tag.decompose()

    # --------------------------------------------------
    # Find the page title
    # --------------------------------------------------

    h1 = soup.find("h1")

    if h1:
        page_title = clean_text(
            h1.get_text(" ", strip=True)
        )
    else:
        page_title = source["title"]

    # --------------------------------------------------
    # Find all headings
    # --------------------------------------------------

    headings = soup.find_all(
        ["h1", "h2", "h3"]
    )

    print(
        f"{source['title']} - headings found:",
        len(headings)
    )

    records = []

    # --------------------------------------------------
    # Extract sections
    # --------------------------------------------------

    for heading in headings:

        section_title = clean_text(
            heading.get_text(" ", strip=True)
        )

        if not section_title:
            continue

        content_parts = []

        # Walk through elements after heading
        # but stop at the next heading.
        for element in heading.find_all_next():

            # Stop when another heading is reached
            if element.name in ["h1", "h2", "h3"]:
                if element != heading:
                    break

            # Extract paragraphs
            if element.name == "p":

                text = clean_text(
                    element.get_text(
                        " ",
                        strip=True
                    )
                )

                if text:
                    content_parts.append(text)

            # Extract list items
            elif element.name == "li":

                text = clean_text(
                    element.get_text(
                        " ",
                        strip=True
                    )
                )

                if text:
                    content_parts.append(text)

            # Extract table cells
            elif element.name in ["td", "th"]:

                text = clean_text(
                    element.get_text(
                        " ",
                        strip=True
                    )
                )

                if text:
                    content_parts.append(text)

        # --------------------------------------------------
        # Combine section content
        # --------------------------------------------------

        content = " ".join(content_parts)

        content = clean_text(content)

        # --------------------------------------------------
        # Remove navigation contamination
        # --------------------------------------------------

        navigation_phrases = [
            "RMIT Europe",
            "RMIT Global",
            "RMIT Vietnam",
            "RMIT UP",
            "RMIT Online",
            "New students",
            "Enrol as a new student",
            "Before semester starts",
            "New research students",
            "My course"
        ]

        # If many navigation phrases occur,
        # discard this record.
        navigation_matches = sum(
            phrase.lower() in content.lower()
            for phrase in navigation_phrases
        )

        if navigation_matches >= 2:
            continue

        # --------------------------------------------------
        # Skip extremely short sections
        # --------------------------------------------------

        if len(content) < 50:
            continue

        records.append({

            "source_id": source_id,

            "category": source["category"],

            "title": page_title,

            "section": section_title,

            "content": content,

            "source_url": source["url"],

            "source_type": source["type"],

            "collection_date": str(date.today())

        })

    return records

In [247]:
all_records = []

for source_id, source in sources.items():

    print("\n" + "=" * 70)
    print("Scraping:", source["title"])
    print("=" * 70)

    try:

        if source["type"] == "policy":

            records = scrape_rmit_policy(
                source_id,
                source
            )

        else:

            records = scrape_rmit_page(
                source_id,
                source
            )

        print(
            "Sections collected:",
            len(records)
        )

        all_records.extend(
            records
        )

    except Exception as e:

        print(
            "ERROR:",
            source["title"]
        )

        print(e)


print("\n" + "=" * 70)

print(
    "TOTAL SECTIONS:",
    len(all_records)
)

print("=" * 70)


Scraping: Extensions
Extensions - headings found: 7
Sections collected: 4

Scraping: Special Consideration
Special Consideration - headings found: 15
Sections collected: 8

Scraping: Equitable Assessment Arrangements
Equitable Assessment Arrangements - headings found: 8
Sections collected: 2

Scraping: Equitable Learning FAQs
Equitable Learning FAQs - headings found: 45
Sections collected: 34

Scraping: Assessment and Assessment Flexibility Policy
Policy headings found: 17
Sections collected: 16

TOTAL SECTIONS: 64


In [248]:
web_df = pd.DataFrame(all_records)

print("Rows:", len(web_df))
print("Columns:", list(web_df.columns))

Rows: 64
Columns: ['source_id', 'category', 'title', 'section', 'content', 'source_url', 'source_type', 'collection_date', 'effective_date']


In [249]:
print(
    web_df.groupby(
        [
            "source_id",
            "category"
        ]
    ).size()
)

source_id  category                         
EAA-01     Equitable Assessment Arrangements     2
EAA-02     Equitable Assessment Arrangements    34
EXT-01     Extension                             4
POL-01     Policy                               16
SC-01      Special Consideration                 8
dtype: int64


In [250]:
display(web_df.head(10))

,source_id,category,title,section,content,source_url,source_type,collection_date,effective_date
0,EXT-01,Extension,Extensions,"If you can't submit an assessment on time due to unforeseen circumstances outside your control, you may be eligible for an extension to the due date.","If you are prevented from submitting an assessment on time due to unexpected/unforeseen circumstances that are outside your control, and are of a short-term nature, you may apply in advance for an extension to the due date of up to seven calendar days . If you need an extension of more than seve...",https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/extensions,webpage,2026-10-03,NaN
1,EXT-01,Extension,Extensions,Assessments eligible for an extension,"You can apply for an extension for assessments which have a deadline, such as assignments, essays and projects, but not for timed assessments such as tests, exams, quizzes or lab/practical assessments. If you want to apply for assessment flexibility for a timed assessment, you must apply for spe...",https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/extensions,webpage,2026-10-03,NaN
2,EXT-01,Extension,Extensions,How to apply,"You must apply at least one working day before the assessment deadline . If you are applying for an extension on or after an assessment due date, you must apply for special consideration . If you have access to Canvas, use the Assessment Extensions Tool . For help using the tool, refer to the St...",https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/extensions,webpage,2026-10-03,NaN
3,EXT-01,Extension,Extensions,False documents and misleading information,"Creating, submitting or using fraudulent documents constitutes a breach of the RMIT Student Charter and may be deemed misconduct under the Student Conduct Policy. Providing RMIT with fraudulent documents is also a crime under the Victorian Crimes Act 1958. Students who submit fraudulent document...",https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/extensions,webpage,2026-10-03,NaN
4,SC-01,Special Consideration,Special consideration,Special consideration,"If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration. If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for spe...",https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/special-consideration,webpage,2026-10-03,NaN
5,SC-01,Special Consideration,Special consideration,"If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration.",What is special consideration? Eligibility How to apply Supporting documentation Technical issues during assessments,https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/special-consideration,webpage,2026-10-03,NaN
6,SC-01,Special Consideration,Special consideration,What is special consideration?,"If unexpected circumstances outside your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration. Special consideration is made available by the University on the understanding that students will use it sparingly and only in cases of p...",https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/special-consideration,webpage,2026-10-03,NaN
7,SC-01,Special Consideration,Special consideration,Eligibility,"If unexpected circumstances outside your control have affected your ability to complete an assessment, then you may be eligible to apply for special consideration. an unexpected short-term physical or mental health condition difficult pers

In [251]:
print(
    web_df.groupby(
        ["source_id", "category"]
    ).size()
)

source_id  category                         
EAA-01     Equitable Assessment Arrangements     2
EAA-02     Equitable Assessment Arrangements    34
EXT-01     Extension                             4
POL-01     Policy                               16
SC-01      Special Consideration                 8
dtype: int64


In [252]:
pd.set_option(
    "display.max_colwidth",
    300
)

display(
    web_df[
        [
            "source_id",
            "section",
            "content"
        ]
    ].head(20)
)

,source_id,section,content
0,EXT-01,"If you can't submit an assessment on time due to unforeseen circumstances outside your control, you may be eligible for an extension to the due date.","If you are prevented from submitting an assessment on time due to unexpected/unforeseen circumstances that are outside your control, and are of a short-term nature, you may apply in advance for an extension to the due date of up to seven calendar days . If you need an extension of more than seve..."
1,EXT-01,Assessments eligible for an extension,"You can apply for an extension for assessments which have a deadline, such as assignments, essays and projects, but not for timed assessments such as tests, exams, quizzes or lab/practical assessments. If you want to apply for assessment flexibility for a timed assessment, you must apply for spe..."
2,EXT-01,How to apply,"You must apply at least one working day before the assessment deadline . If you are applying for an extension on or after an assessment due date, you must apply for special consideration . If you have access to Canvas, use the Assessment Extensions Tool . For help using the tool, refer to the St..."
3,EXT-01,False documents and misleading information,"Creating, submitting or using fraudulent documents constitutes a breach of the RMIT Student Charter and may be deemed misconduct under the Student Conduct Policy. Providing RMIT with fraudulent documents is also a crime under the Victorian Crimes Act 1958. Students who submit fraudulent document..."
4,SC-01,Special consideration,"If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration. If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for spe..."
5,SC-01,"If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration.",What is special consideration? Eligibility How to apply Supporting documentation Technical issues during assessments
6,SC-01,What is special consideration?,"If unexpected circumstances outside your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration. Special consideration is made available by the University on the understanding that students will use it sparingly and only in cases of p..."
7,SC-01,Eligibility,"If unexpected circumstances outside your control have affected your ability to complete an assessment, then you may be eligible to apply for special consideration. an unexpected short-term physical or mental health condition difficult personal circumstances or significant emotional disturbance u..."
8,SC-01,How to apply,Most students can apply for special consideration online (see exceptions below). Please ensure you include a personal statement and supporting documentation with your application. You must apply within five working days after the assessment date or due date . You can submit your application even...
9,SC-01,Supporting documentation,"If you’re applying for special consideration, it is essential that you provide relevant formal, independent and verifiable documentation to support your application. A personal statement alone is not sufficient to support an application for special consideration. You're required to provide forma..."


In [253]:
policy_df = web_df[
    web_df["source_id"] == "POL-01"
].copy()

print(
    "Policy records:",
    len(policy_df)
)

display(
    policy_df[
        [
            "section",
            "content"
        ]
    ]
)

Policy records: 16


,section,content
48,Assessment and Assessment Flexibility Policy,Section 1 - Purpose Section 2 - Scope Section 3 - Policy Design of Assessment Students and Assessment Academic Integrity Responsibilities in Assessment Assessment Flexibility Academic Progress Supplementary Assessment Processes Section 4 - Procedures and Resources Section 5 - Definitions
49,Section 1 - Purpose,"(1) To ensure: Relevant and authentic assessment that supports and enables students to demonstrate evidence of learning at the appropriate level of study. Flexible, equitable and inclusive assessment with a commitment to caring for students whose circumstances require assessment flexibility. Qua..."
50,Section 2 - Scope,"(2) This policy is made pursuant to the Asessment, Academic Progress and Appeals Regulations . Except where otherwise stated it applies to all courses and programs offered by RMIT in the following categories of award: Higher Degree by Research RMIT accredited programs Vocational Education Progra..."
51,Design of Assessment,"(4) Learning outcomes are a central focus for design, quality and standards in assessment. (5) All learning outcomes in a course are specified in the course guide and are assessed. (6) Assessment tasks are appropriately designed to measure student achievement of learning outcomes. (7) Assessment..."
52,Students and Assessment,"(15) Course guides specify all assessment requirements for a course and the weightings of each assessment task, if applicable. (16) Information provided to students on assessment tasks state the performance standards so that students understand the level of attainment required. (17) Students rec..."
53,Academic Integrity,(23) Students demonstrate academic integrity in their assessment practices by: engaging with assessment activities in an honest way providing accountability for the authorship and originality of work submitted acknowledging the work of others and the re-use of original work. (24) Staff take an e...
54,Responsibilities in Assessment,"(27) Deans/Heads of School/Cluster Directors are responsible for: applying university assessment policy and processes in their school, industry cluster or centre; convening course assessment committees and program assessment boards for each program in accordance with the membership and terms of ..."
55,Assessment Flexibility,"(33) The purpose of special consideration, extensions and individual assessment arrangements is to ensure that students experiencing extenuating circumstances or who have specific needs are appropriately supported and can seek assessment arrangements that provide the best circumstances for ongoi..."
56,Extensions,(34) Extensions are available for unforeseen circumstances of a short-term nature. (35) Applications are submitted to the school/industry cluster at least one working day before the due date for an assessment. (36) Extensions can be approved for up to one week (seven calendar days) after the due...
57,Special Consideration,"(41) Special consideration is available for unexpected circumstances outside students’ control. These include but are not limited to unexpected short-term ill health, and unavoidable family, work, cultural or religious commitments. (42) An application for special consideration is made in advance..."


In [254]:
clean_df = web_df.copy()

# Normalize whitespace
clean_df["content"] = (
    clean_df["content"]
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
)

# Find duplicates
duplicate_mask = clean_df.duplicated(
    subset=["content"],
    keep=False
)

print(
    "Records involved in duplicate content:",
    duplicate_mask.sum()
)

Records involved in duplicate content: 0


In [255]:
display(
    clean_df.loc[
        duplicate_mask,
        [
            "source_id",
            "section",
            "content"
        ]
    ]
)

,source_id,section,content


In [256]:
raw_path = "./rmit_assessment_support_scraped_raw.csv"

web_df.to_csv(
    raw_path,
    index=False,
    encoding="utf-8-sig"
)

print("Raw dataset saved:")
print(raw_path)

Raw dataset saved:
./rmit_assessment_support_scraped_raw.csv


In [257]:
navigation_phrases = [
    "RMIT Global",
    "RMIT Vietnam",
    "RMIT UP",
    "RMIT Online",
    "New students",
    "Enrol as a new student",
    "Before semester starts",
    "New research students",
    "My course"
]

for phrase in navigation_phrases:

    count = web_df["content"].str.contains(
        phrase,
        case=False,
        na=False
    ).sum()

    print(
        f"{phrase} → {count}"
    )

RMIT Global → 0
RMIT Vietnam → 0
RMIT UP → 2
RMIT Online → 0
New students → 0
Enrol as a new student → 0
Before semester starts → 0
New research students → 0
My course → 0


In [258]:
duplicate_mask = web_df.duplicated(
    subset=["content"],
    keep=False
)

print(
    "Records involved in duplicate content:",
    duplicate_mask.sum()
)

Records involved in duplicate content: 0


In [259]:
display(
    web_df.loc[
        duplicate_mask,
        [
            "source_id",
            "section",
            "content"
        ]
    ]
)

,source_id,section,content


In [260]:
clean_df = web_df.copy()

# Normalize content
clean_df["content"] = (
    clean_df["content"]
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
)

# Remove exact duplicate content
before = len(clean_df)

clean_df = clean_df.drop_duplicates(
    subset=["content"],
    keep="first"
).reset_index(drop=True)

after = len(clean_df)

print(
    "Records before:",
    before
)

print(
    "Duplicate records removed:",
    before - after
)

print(
    "Records after:",
    after
)

Records before: 64
Duplicate records removed: 0
Records after: 64


In [261]:
def contains_navigation(text):

    text = str(text).lower()

    navigation_phrases = [
        "rmit europe",
        "rmit global",
        "rmit vietnam",
        "rmit up",
        "rmit online",
        "new students",
        "enrol as a new student",
        "before semester starts",
        "new research students"
    ]

    matches = sum(
        phrase in text
        for phrase in navigation_phrases
    )

    return matches >= 2

In [262]:
navigation_mask = clean_df["content"].apply(
    contains_navigation
)

print(
    "Navigation records found:",
    navigation_mask.sum()
)

Navigation records found: 0


In [263]:
before = len(clean_df)

clean_df = clean_df.drop_duplicates(
    subset=["content"],
    keep="first"
).reset_index(drop=True)

after = len(clean_df)

print(
    "Records before:",
    before
)

print(
    "Duplicates removed:",
    before - after
)

print(
    "Records after:",
    after
)

Records before: 64
Duplicates removed: 0
Records after: 64


In [264]:
clean_df = clean_df[
    ~navigation_mask
].reset_index(drop=True)

print(
    "Final clean records:",
    len(clean_df)
)

Final clean records: 64


In [265]:
print(
    clean_df.isnull().sum()
)

source_id           0
category            0
title               0
section             0
content             0
source_url          0
source_type         0
collection_date     0
effective_date     48
dtype: int64


In [266]:
clean_df["content_length"] = (
    clean_df["content"]
    .str.len()
)

print(
    clean_df["content_length"].describe()
)

count       64.000000
mean      1201.250000
std       1776.174702
min        101.000000
25%        315.750000
50%        591.500000
75%       1284.250000
max      12073.000000
Name: content_length, dtype: float64


In [267]:
display(
    clean_df[
        clean_df["content_length"] < 100
    ][
        [
            "source_id",
            "section",
            "content",
            "content_length"
        ]
    ]
)

,source_id,section,content,content_length


In [268]:
clean_df["document_version"] = np.where(
    clean_df["source_id"] == "POL-01",
    "Current - effective 28 July 2026",
    "Current webpage"
)

In [269]:
print("\nMissing values:")
print(clean_df.isnull().sum())


Missing values:
source_id            0
category             0
title                0
section              0
content              0
source_url           0
source_type          0
collection_date      0
effective_date      48
content_length       0
document_version     0
dtype: int64


In [270]:
clean_df["effective_date"] = clean_df["effective_date"].fillna(
    "Not specified"
)

print(clean_df["effective_date"].value_counts())

effective_date
Not specified    48
2026-07-28       16
Name: count, dtype: int64


In [271]:
clean_path = (
    "./"
    "rmit_assessment_support_scraped_cleaned.csv"
)

clean_df.to_csv(
    clean_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Clean CSV saved:"
)

print(clean_path)

Clean CSV saved:
./rmit_assessment_support_scraped_cleaned.csv


In [272]:
print(
    "FINAL DATASET"
)

print(
    "Rows:",
    len(clean_df)
)

print(
    "Columns:",
    list(clean_df.columns)
)

display(
    clean_df[
        [
            "source_id",
            "category",
            "section",
            "content"
        ]
    ]
)

FINAL DATASET
Rows: 64
Columns: ['source_id', 'category', 'title', 'section', 'content', 'source_url', 'source_type', 'collection_date', 'effective_date', 'content_length', 'document_version']


,source_id,category,section,content
0,EXT-01,Extension,"If you can't submit an assessment on time due to unforeseen circumstances outside your control, you may be eligible for an extension to the due date.","If you are prevented from submitting an assessment on time due to unexpected/unforeseen circumstances that are outside your control, and are of a short-term nature, you may apply in advance for an extension to the due date of up to seven calendar days . If you need an extension of more than seve..."
1,EXT-01,Extension,Assessments eligible for an extension,"You can apply for an extension for assessments which have a deadline, such as assignments, essays and projects, but not for timed assessments such as tests, exams, quizzes or lab/practical assessments. If you want to apply for assessment flexibility for a timed assessment, you must apply for spe..."
2,EXT-01,Extension,How to apply,"You must apply at least one working day before the assessment deadline . If you are applying for an extension on or after an assessment due date, you must apply for special consideration . If you have access to Canvas, use the Assessment Extensions Tool . For help using the tool, refer to the St..."
3,EXT-01,Extension,False documents and misleading information,"Creating, submitting or using fraudulent documents constitutes a breach of the RMIT Student Charter and may be deemed misconduct under the Student Conduct Policy. Providing RMIT with fraudulent documents is also a crime under the Victorian Crimes Act 1958. Students who submit fraudulent document..."
4,SC-01,Special Consideration,Special consideration,"If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration. If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for spe..."
5,SC-01,Special Consideration,"If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration.",What is special consideration? Eligibility How to apply Supporting documentation Technical issues during assessments
6,SC-01,Special Consideration,What is special consideration?,"If unexpected circumstances outside your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration. Special consideration is made available by the University on the understanding that students will use it sparingly and only in cases of p..."
7,SC-01,Special Consideration,Eligibility,"If unexpected circumstances outside your control have affected your ability to complete an assessment, then you may be eligible to apply for special consideration. an unexpected short-term physical or mental health condition difficult personal circumstances or significant emotional disturbance u..."
8,SC-01,Special Consideration,How to apply,Most students can apply for special consideration online (see exceptions below). Please ensure you include a personal statement and supporting documentation with your application. You must apply within five working days after the assessment date or due date . You can submit your application even...
9,SC-01,Special Consideration,Supporting documentation,"If you’re applying for special consideration, it is essential that you provide relevant formal, independent and verifiable documentation to support your application. A personal statement alone is not sufficient to support an application for special consideration. You're required to provide forma..."


In [273]:
web_df["effective_date"] = web_df["effective_date"].fillna(
    "Not specified"
)

print(web_df["effective_date"].value_counts())

effective_date
Not specified    48
2026-07-28       16
Name: count, dtype: int64


In [274]:
web_df["effective_date"] = web_df["effective_date"].fillna(
    "Not specified"
)

print(web_df["effective_date"].value_counts())

effective_date
Not specified    48
2026-07-28       16
Name: count, dtype: int64


In [275]:
print("\nMissing values:")
print(web_df.isnull().sum())


Missing values:
source_id          0
category           0
title              0
section            0
content            0
source_url         0
source_type        0
collection_date    0
effective_date     0
dtype: int64


In [276]:
large_records = clean_df[
    clean_df["content_length"] > 3000
][
    ["source_id", "category", "section", "content_length"]
].sort_values(
    "content_length",
    ascending=False
)

display(large_records)

,source_id,category,section,content_length
10,SC-01,Special Consideration,False documents and misleading information,12073
63,POL-01,Policy,Section 5 - Definitions,5615
22,EAA-02,Equitable Assessment Arrangements,Top tips for students seeking ELS support,4140
8,SC-01,Special Consideration,How to apply,3651
58,POL-01,Policy,Equitable Assessment Arrangements,3483
6,SC-01,Special Consideration,What is special consideration?,3222


In [277]:
print(clean_df.shape)
print(clean_df.columns.tolist())

(64, 11)
['source_id', 'category', 'title', 'section', 'content', 'source_url', 'source_type', 'collection_date', 'effective_date', 'content_length', 'document_version']


In [278]:
def chunk_text(text, chunk_size=1000):
    text = str(text).strip()

    # Split into complete sentences
    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current_sentences = []
    current_length = 0

    for sentence in sentences:
        sentence = sentence.strip()

        if not sentence:
            continue

        # If adding this sentence keeps us within the target size
        if current_length + len(sentence) <= chunk_size:
            current_sentences.append(sentence)
            current_length += len(sentence) + 1

        else:
            # Save current chunk
            if current_sentences:
                chunks.append(" ".join(current_sentences))

            # Use the last complete sentence as overlap
            if current_sentences:
                current_sentences = [current_sentences[-1], sentence]
                current_length = len(current_sentences[-2]) + len(sentence) + 1
            else:
                current_sentences = [sentence]
                current_length = len(sentence)

    # Save final chunk
    if current_sentences:
        chunks.append(" ".join(current_sentences))

    return chunks

In [279]:
chunk_rows = []

for _, row in clean_df.iterrows():
    
    chunks = chunk_text(row["content"])
    
    for chunk_number, chunk in enumerate(chunks, start=1):
        
        chunk_rows.append({
            "source_id": row["source_id"],
            "category": row["category"],
            "title": row["title"],
            "section": row["section"],
            "chunk_id": f'{row["source_id"]}_{chunk_number:03d}',
            "chunk_number": chunk_number,
            "content": chunk,
            "source_url": row["source_url"],
            "source_type": row["source_type"],
            "collection_date": row["collection_date"],
            "effective_date": row["effective_date"],
            "document_version": row["document_version"]
        })

chunk_df = pd.DataFrame(chunk_rows)

print("Original records:", len(clean_df))
print("Total chunks:", len(chunk_df))

display(chunk_df.head(10))

Original records: 64
Total chunks: 127


,source_id,category,title,section,chunk_id,chunk_number,content,source_url,source_type,collection_date,effective_date,document_version
0,EXT-01,Extension,Extensions,"If you can't submit an assessment on time due to unforeseen circumstances outside your control, you may be eligible for an extension to the due date.",EXT-01_001,1,"If you are prevented from submitting an assessment on time due to unexpected/unforeseen circumstances that are outside your control, and are of a short-term nature, you may apply in advance for an extension to the due date of up to seven calendar days . If you need an extension of more than seve...",https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/extensions,webpage,2026-10-03,Not specified,Current webpage
1,EXT-01,Extension,Extensions,Assessments eligible for an extension,EXT-01_001,1,"You can apply for an extension for assessments which have a deadline, such as assignments, essays and projects, but not for timed assessments such as tests, exams, quizzes or lab/practical assessments. If you want to apply for assessment flexibility for a timed assessment, you must apply for spe...",https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/extensions,webpage,2026-10-03,Not specified,Current webpage
2,EXT-01,Extension,Extensions,How to apply,EXT-01_001,1,"You must apply at least one working day before the assessment deadline . If you are applying for an extension on or after an assessment due date, you must apply for special consideration . If you have access to Canvas, use the Assessment Extensions Tool . For help using the tool, refer to the St...",https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/extensions,webpage,2026-10-03,Not specified,Current webpage
3,EXT-01,Extension,Extensions,How to apply,EXT-01_002,2,"download and complete the Application for Extension of Time for Submission of Assessable Work Form (PDF, 149KB) submit it to the RMIT School that runs your course. Check whether the form needs to be submitted to the School Administrative Officer, Course Coordinator (higher education), Program Ma...",https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/extensions,webpage,2026-10-03,Not specified,Current webpage
4,EXT-01,Extension,Extensions,False documents and misleading information,EXT-01_001,1,"Creating, submitting or using fraudulent documents constitutes a breach of the RMIT Student Charter and may be deemed misconduct under the Student Conduct Policy. Providing RMIT with fraudulent documents is also a crime under the Victorian Crimes Act 1958. Students who submit fraudulent document...",https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/extensions,webpage,2026-10-03,Not specified,Current webpage
5,SC-01,Special Consideration,Special consideration,Special consideration,SC-01_001,1,"If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration. If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for spe...",https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/special-consideration,webpage,2026-10-03,Not specified,Current webpage
6,SC-01,Special Consideration,Special consideration,"If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration.",SC-01_001,1,What is special consideration? Eligibility How to apply Supporting documentation Technical issues during assessments,https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/special-consideration,webpage,2026-10-03,Not specified,Current webpage
7,SC-01,Special Consideration,Special conside

In [280]:
print("Original records:", len(clean_df))
print("Total chunks:", len(chunk_df))
print("Unique source IDs:", chunk_df["source_id"].nunique())

print("\nChunk length statistics:")
print(chunk_df["content"].str.len().describe())

Original records: 64
Total chunks: 127
Unique source IDs: 5

Chunk length statistics:
count    127.000000
mean     678.228346
std      277.401554
min      101.000000
25%      410.000000
50%      787.000000
75%      924.500000
max      999.000000
Name: content, dtype: float64


In [281]:
test_text = clean_df.loc[10, "content"]

test_chunks = chunk_text(test_text)

print("Original length:", len(test_text))
print("Number of chunks:", len(test_chunks))

for i, chunk in enumerate(test_chunks[:3]):
    print("\n--- CHUNK", i + 1, "---")
    print(chunk)

Original length: 12073
Number of chunks: 16

--- CHUNK 1 ---
Creating, submitting or using fraudulent documents constitutes a breach of the RMIT Student Charter and may be deemed misconduct under the Student Conduct Policy. Providing RMIT with fraudulent documents is also a crime under the Victorian Crimes Act 1958. Students who submit fraudulent documents may face a range of consequences, from failing an assessment task through to permanent expulsion or revocation of qualification/s. Find out more about document fraud , how you can avoid it and where you can access free support if you need help navigating a challenging situation. Different documents may be required depending on the reasons for your application. Reason Supporting documents An unexpected short-term physical or mental health condition It’s recommended you download, complete and include an Impact Assessment Statement (PDF, 45KB) completed by your treating health practitioner.

--- CHUNK 2 ---
Reason Supporting documents A

In [282]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully")

Embedding model loaded successfully


In [283]:
chunk_embeddings = embedding_model.encode(
    chunk_df["content"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Number of chunks:", len(chunk_df))
print("Embedding shape:", chunk_embeddings.shape)

Batches: 100%|██████████| 4/4 [00:03<00:00,  1.21it/s]

Number of chunks: 127
Embedding shape: (127, 384)


In [284]:
import faiss
import numpy as np

# FAISS works with float32
embeddings = chunk_embeddings.astype("float32")

# Normalise vectors so we can use cosine similarity
faiss.normalize_L2(embeddings)

# Each embedding has 384 dimensions
dimension = embeddings.shape[1]

# Create the FAISS search index
index = faiss.IndexFlatIP(dimension)

# Add all chunk embeddings to the index
index.add(embeddings)

print("FAISS index created successfully")
print("Embedding dimensions:", dimension)
print("Vectors stored in index:", index.ntotal)

FAISS index created successfully
Embedding dimensions: 384
Vectors stored in index: 127


In [285]:
question = "Can I get an extension if I am sick?"

# Convert the question into an embedding
question_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

# Normalise it in the same way as our document embeddings
faiss.normalize_L2(question_embedding)

# Retrieve the 5 most relevant chunks
scores, indices = index.search(question_embedding, 5)

print("Question:", question)

for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    row = chunk_df.iloc[idx]

    print("\n------------------------------")
    print("RESULT", rank)
    print("Similarity score:", round(float(score), 4))
    print("Source:", row["source_id"])
    print("Category:", row["category"])
    print("Section:", row["section"])
    print("Chunk ID:", row["chunk_id"])
    print("\nContent:")
    print(row["content"])

Question: Can I get an extension if I am sick?

------------------------------
RESULT 1
Similarity score: 0.431
Source: EAA-02
Category: Equitable Assessment Arrangements
Section: How do I ask for an extension?
Chunk ID: EAA-02_001

Content:
If you have the Equitable Assessment Arrangement for ‘Extensions’ within your Equitable Learning Plan, please ask your teacher how you should apply for an extension before the adjustment is due. Different courses use different extension application processes, so you may be required to: Apply for an extension via Canvas Apply for an extension via email to your teacher or Course Coordinator or Complete and submit an extension application form Please note, this adjustment only applies to individual assessments and extensions must be requested at least one business/working day before the deadline.You cannot seek an extension if the due date has passed. Clearly ask for an extension and suggest a reasonable new due date that considers the impacts of your

In [286]:
display(clean_df[["source_id", "category", "title", "section"]])

,source_id,category,title,section
0,EXT-01,Extension,Extensions,"If you can't submit an assessment on time due to unforeseen circumstances outside your control, you may be eligible for an extension to the due date."
1,EXT-01,Extension,Extensions,Assessments eligible for an extension
2,EXT-01,Extension,Extensions,How to apply
3,EXT-01,Extension,Extensions,False documents and misleading information
4,SC-01,Special Consideration,Special consideration,Special consideration
5,SC-01,Special Consideration,Special consideration,"If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration."
6,SC-01,Special Consideration,Special consideration,What is special consideration?
7,SC-01,Special Consideration,Special consideration,Eligibility
8,SC-01,Special Consideration,Special consideration,How to apply
9,SC-01,Special Consideration,Special consideration,Supporting documentation


In [287]:
pd.set_option("display.max_rows", None)

display(
    clean_df[["source_id", "category", "section"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

,source_id,category,section
0,EXT-01,Extension,"If you can't submit an assessment on time due to unforeseen circumstances outside your control, you may be eligible for an extension to the due date."
1,EXT-01,Extension,Assessments eligible for an extension
2,EXT-01,Extension,How to apply
3,EXT-01,Extension,False documents and misleading information
4,SC-01,Special Consideration,Special consideration
5,SC-01,Special Consideration,"If unexpected circumstances beyond your control have affected your ability to complete an assessment, you may be eligible to apply for special consideration."
6,SC-01,Special Consideration,What is special consideration?
7,SC-01,Special Consideration,Eligibility
8,SC-01,Special Consideration,How to apply
9,SC-01,Special Consideration,Supporting documentation


In [288]:
test_questions = [
    {
        "question": "Can I get an extension for an assignment?",
        "expected_source": "EXT-01"
    },
    {
        "question": "What should I do if I need more time because of unexpected circumstances?",
        "expected_source": "EXT-01"
    },
    {
        "question": "Am I eligible for special consideration?",
        "expected_source": "SC-01"
    },
    {
        "question": "What evidence do I need for special consideration?",
        "expected_source": "SC-01"
    },
    {
        "question": "How do I apply for an Equitable Assessment Arrangement?",
        "expected_source": "EAA-01"
    },
    {
        "question": "How do I ask for an extension using my Equitable Learning Plan?",
        "expected_source": "EAA-02"
    },

    {
    "question": "Can I get an extension for an exam?",
    "expected_source": "EXT-01"
    },
    {
    "question": "How many days can an assessment extension be?",
    "expected_source": "EXT-01"
    },
    {
    "question": "What should I do if I have technical problems during an online assessment?",
    "expected_source": "SC-01"
    },
    {
    "question": "What documents do I need when registering for disability support?",
    "expected_source": "EAA-02"
    },
    {
    "question": "Can I get an extension for a group assignment?",
    "expected_source": "EAA-02"
    },
    {
    "question": "Can assessment arrangements apply to online tests and practicals?",
    "expected_source": "EAA-02"
    }


]

test_df = pd.DataFrame(test_questions)

display(test_df)

,question,expected_source
0,Can I get an extension for an assignment?,EXT-01
1,What should I do if I need more time because of unexpected circumstances?,EXT-01
2,Am I eligible for special consideration?,SC-01
3,What evidence do I need for special consideration?,SC-01
4,How do I apply for an Equitable Assessment Arrangement?,EAA-01
5,How do I ask for an extension using my Equitable Learning Plan?,EAA-02
6,Can I get an extension for an exam?,EXT-01
7,How many days can an assessment extension be?,EXT-01
8,What should I do if I have technical problems during an online assessment?,SC-01
9,What documents do I need when registering for disability support?,EAA-02


In [289]:
results = []

for _, test in test_df.iterrows():

    # Convert the question into an embedding
    query_embedding = embedding_model.encode(
        [test["question"]],
        convert_to_numpy=True
    ).astype("float32")

    # Normalise the question embedding
    faiss.normalize_L2(query_embedding)

    # Retrieve the single most relevant chunk
    scores, indices = index.search(query_embedding, 1)

    top_index = indices[0][0]
    top_score = scores[0][0]

    # Find which source the retrieved chunk came from
    retrieved_row = chunk_df.iloc[top_index]
    retrieved_source = retrieved_row["source_id"]

    # Compare retrieved source with expected source
    correct = retrieved_source == test["expected_source"]

    results.append({
        "question": test["question"],
        "expected_source": test["expected_source"],
        "retrieved_source": retrieved_source,
        "score": round(float(top_score), 4),
        "correct": correct
    })

evaluation_df = pd.DataFrame(results)

display(evaluation_df)

,question,expected_source,retrieved_source,score,correct
0,Can I get an extension for an assignment?,EXT-01,EAA-02,0.5647,False
1,What should I do if I need more time because of unexpected circumstances?,EXT-01,EAA-02,0.3558,False
2,Am I eligible for special consideration?,SC-01,SC-01,0.7528,True
3,What evidence do I need for special consideration?,SC-01,SC-01,0.5690,True
4,How do I apply for an Equitable Assessment Arrangement?,EAA-01,POL-01,0.7520,False
5,How do I ask for an extension using my Equitable Learning Plan?,EAA-02,EAA-02,0.8096,True
6,Can I get an extension for an exam?,EXT-01,EAA-02,0.6186,False
7,How many days can an assessment extension be?,EXT-01,EXT-01,0.8408,True
8,What should I do if I have technical problems during an online assessment?,SC-01,SC-01,0.7421,True
9,What documents do I need when registering for disability support?,EAA-02,EAA-02,0.7236,True


In [290]:
results_top3 = []

for _, test in test_df.iterrows():

    query_embedding = embedding_model.encode(
        [test["question"]],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    # Retrieve top 3 chunks
    scores, indices = index.search(query_embedding, 3)

    retrieved_sources = [
        chunk_df.iloc[idx]["source_id"]
        for idx in indices[0]
    ]

    # Is the expected source anywhere in the top 3?
    correct_top3 = test["expected_source"] in retrieved_sources

    results_top3.append({
        "question": test["question"],
        "expected_source": test["expected_source"],
        "top_3_sources": retrieved_sources,
        "correct_top3": correct_top3
    })

evaluation_top3_df = pd.DataFrame(results_top3)

display(evaluation_top3_df)

,question,expected_source,top_3_sources,correct_top3
0,Can I get an extension for an assignment?,EXT-01,"[EAA-02, EAA-02, EAA-02]",False
1,What should I do if I need more time because of unexpected circumstances?,EXT-01,"[EAA-02, SC-01, SC-01]",False
2,Am I eligible for special consideration?,SC-01,"[SC-01, SC-01, SC-01]",True
3,What evidence do I need for special consideration?,SC-01,"[SC-01, SC-01, SC-01]",True
4,How do I apply for an Equitable Assessment Arrangement?,EAA-01,"[POL-01, POL-01, EAA-01]",True
5,How do I ask for an extension using my Equitable Learning Plan?,EAA-02,"[EAA-02, EAA-02, EAA-02]",True
6,Can I get an extension for an exam?,EXT-01,"[EAA-02, EAA-02, EXT-01]",True
7,How many days can an assessment extension be?,EXT-01,"[EXT-01, POL-01, EAA-02]",True
8,What should I do if I have technical problems during an online assessment?,SC-01,"[SC-01, SC-01, SC-01]",True
9,What documents do I need when registering for disability support?,EAA-02,"[EAA-02, SC-01, SC-01]",True


In [291]:
question = "Can I get an extension for an assignment?"

query_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(query_embedding)

# Retrieve top 5
scores, indices = index.search(query_embedding, 5)

print("Question:", question)

for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    row = chunk_df.iloc[idx]

    print("\n------------------------------")
    print("Rank:", rank)
    print("Source:", row["source_id"])
    print("Section:", row["section"])
    print("Score:", round(float(score), 4))
    print("Content:", row["content"][:400])

Question: Can I get an extension for an assignment?

------------------------------
Rank: 1
Source: EAA-02
Section: My teacher has directed me to apply for special consideration even though I have the Equitable Assessment Arrangement for extensions. What do I do?
Score: 0.5647
Content: If it is your first extension, you do not need to apply for special consideration. Direct your teacher to your Equitable Learning Plan where this is clearly stated. If you aren’t able to implement your extension request, contact ELS for assistance. If you require a second extension, you will need to apply for special consideration .

------------------------------
Rank: 2
Source: EAA-02
Section: How do I ask for an extension?
Score: 0.5367
Content: The most common extension is seven days for an individual assessment. For group assessments, you can request an extension of up to seven days without having to use your ELP and disclose you have adjustments to other group members. You can only request one exte

In [292]:
results_top5 = []

for _, test in test_df.iterrows():

    query_embedding = embedding_model.encode(
        [test["question"]],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    # Retrieve top 5 chunks
    scores, indices = index.search(query_embedding, 5)

    retrieved_sources = [
        chunk_df.iloc[idx]["source_id"]
        for idx in indices[0]
    ]

    correct_top5 = test["expected_source"] in retrieved_sources

    results_top5.append({
        "question": test["question"],
        "expected_source": test["expected_source"],
        "top_5_sources": retrieved_sources,
        "correct_top5": correct_top5
    })

evaluation_top5_df = pd.DataFrame(results_top5)

display(evaluation_top5_df)

,question,expected_source,top_5_sources,correct_top5
0,Can I get an extension for an assignment?,EXT-01,"[EAA-02, EAA-02, EAA-02, EAA-02, EXT-01]",True
1,What should I do if I need more time because of unexpected circumstances?,EXT-01,"[EAA-02, SC-01, SC-01, EAA-02, EXT-01]",True
2,Am I eligible for special consideration?,SC-01,"[SC-01, SC-01, SC-01, POL-01, SC-01]",True
3,What evidence do I need for special consideration?,SC-01,"[SC-01, SC-01, SC-01, SC-01, SC-01]",True
4,How do I apply for an Equitable Assessment Arrangement?,EAA-01,"[POL-01, POL-01, EAA-01, POL-01, POL-01]",True
5,How do I ask for an extension using my Equitable Learning Plan?,EAA-02,"[EAA-02, EAA-02, EAA-02, EAA-02, EAA-02]",True
6,Can I get an extension for an exam?,EXT-01,"[EAA-02, EAA-02, EXT-01, EAA-02, EAA-02]",True
7,How many days can an assessment extension be?,EXT-01,"[EXT-01, POL-01, EAA-02, EAA-02, EAA-02]",True
8,What should I do if I have technical problems during an online assessment?,SC-01,"[SC-01, SC-01, SC-01, SC-01, SC-01]",True
9,What documents do I need when registering for disability support?,EAA-02,"[EAA-02, SC-01, SC-01, EAA-02, SC-01]",True


In [293]:
top1_accuracy = evaluation_df["correct"].mean()
top3_accuracy = evaluation_top3_df["correct_top3"].mean()
top5_accuracy = evaluation_top5_df["correct_top5"].mean()

print("Retrieval Evaluation")
print("--------------------")
print(f"Top-1 Accuracy: {top1_accuracy:.1%}")
print(f"Top-3 Accuracy: {top3_accuracy:.1%}")
print(f"Top-5 Accuracy: {top5_accuracy:.1%}")

Retrieval Evaluation
--------------------
Top-1 Accuracy: 66.7%
Top-3 Accuracy: 83.3%
Top-5 Accuracy: 100.0%


In [294]:
chunk_df["embedding_text"] = (
    chunk_df["category"].fillna("") + ". " +
    chunk_df["section"].fillna("") + ". " +
    chunk_df["content"].fillna("")
)

print("Original content:")
print(chunk_df.iloc[0]["content"][:500])

print("\n--------------------")
print("New embedding text:")
print(chunk_df.iloc[0]["embedding_text"][:700])

Original content:
If you are prevented from submitting an assessment on time due to unexpected/unforeseen circumstances that are outside your control, and are of a short-term nature, you may apply in advance for an extension to the due date of up to seven calendar days . If you need an extension of more than seven days , you must apply for special consideration . However, if you have an equitable assessment arrangement which allows for the negotiation of submission dates with academic/teaching staff, extensions o

--------------------
New embedding text:
Extension. If you can't submit an assessment on time due to unforeseen circumstances outside your control, you may be eligible for an extension to the due date.. If you are prevented from submitting an assessment on time due to unexpected/unforeseen circumstances that are outside your control, and are of a short-term nature, you may apply in advance for an extension to the due date of up to seven calendar days . If you need an extensio

In [295]:
improved_embeddings = embedding_model.encode(
    chunk_df["embedding_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

print("Number of chunks:", len(chunk_df))
print("Improved embedding shape:", improved_embeddings.shape)

Batches: 100%|██████████| 4/4 [00:03<00:00,  1.14it/s]

Number of chunks: 127
Improved embedding shape: (127, 384)


In [296]:
# Make a copy before normalising
improved_embeddings_normalized = improved_embeddings.copy()

# Normalise for cosine similarity
faiss.normalize_L2(improved_embeddings_normalized)

# Create a new FAISS index
dimension = improved_embeddings_normalized.shape[1]
improved_index = faiss.IndexFlatIP(dimension)

# Add improved embeddings
improved_index.add(improved_embeddings_normalized)

print("Improved FAISS index created successfully")
print("Embedding dimensions:", dimension)
print("Vectors stored:", improved_index.ntotal)

Improved FAISS index created successfully
Embedding dimensions: 384
Vectors stored: 127


In [297]:
def evaluate_index(search_index, test_df):
    results = []

    for _, test in test_df.iterrows():

        query_embedding = embedding_model.encode(
            [test["question"]],
            convert_to_numpy=True
        ).astype("float32")

        faiss.normalize_L2(query_embedding)

        # Retrieve top 5
        scores, indices = search_index.search(query_embedding, 5)

        retrieved_sources = [
            chunk_df.iloc[idx]["source_id"]
            for idx in indices[0]
        ]

        results.append({
            "question": test["question"],
            "expected_source": test["expected_source"],
            "top_1_source": retrieved_sources[0],
            "correct_top1": test["expected_source"] == retrieved_sources[0],
            "correct_top3": test["expected_source"] in retrieved_sources[:3],
            "correct_top5": test["expected_source"] in retrieved_sources
        })

    return pd.DataFrame(results)


# Evaluate both versions
baseline_12 = evaluate_index(index, test_df)
improved_12 = evaluate_index(improved_index, test_df)

print("BASELINE - Content Only")
print("-----------------------")
print(f"Top-1: {baseline_12['correct_top1'].mean():.1%}")
print(f"Top-3: {baseline_12['correct_top3'].mean():.1%}")
print(f"Top-5: {baseline_12['correct_top5'].mean():.1%}")

print("\nIMPROVED - Category + Section + Content")
print("---------------------------------------")
print(f"Top-1: {improved_12['correct_top1'].mean():.1%}")
print(f"Top-3: {improved_12['correct_top3'].mean():.1%}")
print(f"Top-5: {improved_12['correct_top5'].mean():.1%}")

BASELINE - Content Only
-----------------------
Top-1: 66.7%
Top-3: 83.3%
Top-5: 100.0%

IMPROVED - Category + Section + Content
---------------------------------------
Top-1: 91.7%
Top-3: 91.7%
Top-5: 91.7%


In [298]:
failed_improved = improved_12[
    improved_12["correct_top1"] == False
]

display(failed_improved[
    ["question", "expected_source", "top_1_source"]
])

,question,expected_source,top_1_source
4,How do I apply for an Equitable Assessment Arrangement?,EAA-01,POL-01


In [299]:
improved_results = []

for _, test in test_df.iterrows():

    # Convert question into embedding
    query_embedding = embedding_model.encode(
        [test["question"]],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    # Retrieve top 5 from improved index
    scores, indices = improved_index.search(query_embedding, 5)

    retrieved_sources = [
        chunk_df.iloc[idx]["source_id"]
        for idx in indices[0]
    ]

    improved_results.append({
        "question": test["question"],
        "expected_source": test["expected_source"],
        "top_1_source": retrieved_sources[0],
        "top_3_sources": retrieved_sources[:3],
        "top_5_sources": retrieved_sources,
        "correct_top1": test["expected_source"] == retrieved_sources[0],
        "correct_top3": test["expected_source"] in retrieved_sources[:3],
        "correct_top5": test["expected_source"] in retrieved_sources
    })

improved_evaluation_df = pd.DataFrame(improved_results)

display(improved_evaluation_df)

print("\nImproved Retrieval Evaluation")
print("-----------------------------")
print(f"Top-1 Accuracy: {improved_evaluation_df['correct_top1'].mean():.1%}")
print(f"Top-3 Accuracy: {improved_evaluation_df['correct_top3'].mean():.1%}")
print(f"Top-5 Accuracy: {improved_evaluation_df['correct_top5'].mean():.1%}")

,question,expected_source,top_1_source,top_3_sources,top_5_sources,correct_top1,correct_top3,correct_top5
0,Can I get an extension for an assignment?,EXT-01,EXT-01,"[EXT-01, EAA-02, EAA-02]","[EXT-01, EAA-02, EAA-02, EAA-02, EXT-01]",True,True,True
1,What should I do if I need more time because of unexpected circumstances?,EXT-01,EXT-01,"[EXT-01, SC-01, SC-01]","[EXT-01, SC-01, SC-01, SC-01, SC-01]",True,True,True
2,Am I eligible for special consideration?,SC-01,SC-01,"[SC-01, SC-01, SC-01]","[SC-01, SC-01, SC-01, SC-01, SC-01]",True,True,True
3,What evidence do I need for special consideration?,SC-01,SC-01,"[SC-01, SC-01, SC-01]","[SC-01, SC-01, SC-01, SC-01, SC-01]",True,True,True
4,How do I apply for an Equitable Assessment Arrangement?,EAA-01,POL-01,"[POL-01, POL-01, POL-01]","[POL-01, POL-01, POL-01, POL-01, EAA-02]",False,False,False
5,How do I ask for an extension using my Equitable Learning Plan?,EAA-02,EAA-02,"[EAA-02, EAA-02, EAA-02]","[EAA-02, EAA-02, EAA-02, EAA-02, EAA-02]",True,True,True
6,Can I get an extension for an exam?,EXT-01,EXT-01,"[EXT-01, EAA-02, EXT-01]","[EXT-01, EAA-02, EXT-01, EAA-02, EXT-01]",True,True,True
7,How many days can an assessment extension be?,EXT-01,EXT-01,"[EXT-01, EAA-02, POL-01]","[EXT-01, EAA-02, POL-01, EXT-01, EAA-02]",True,True,True
8,What should I do if I have technical problems during an online assessment?,SC-01,SC-01,"[SC-01, SC-01, SC-01]","[SC-01, SC-01, SC-01, SC-01, SC-01]",True,True,True
9,What documents do I need when registering for disability support?,EAA-02,EAA-02,"[EAA-02, SC-01, SC-01]","[EAA-02, SC-01, SC-01, EAA-02, EAA-02]",True,True,True



Improved Retrieval Evaluation
-----------------------------
Top-1 Accuracy: 91.7%
Top-3 Accuracy: 91.7%
Top-5 Accuracy: 91.7%


In [300]:
question = "How do I apply for an Equitable Assessment Arrangement?"

query_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(query_embedding)

scores, indices = improved_index.search(query_embedding, 10)

print("Question:", question)

for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    row = chunk_df.iloc[idx]

    print("\n------------------------------")
    print("Rank:", rank)
    print("Source:", row["source_id"])
    print("Category:", row["category"])
    print("Section:", row["section"])
    print("Score:", round(float(score), 4))

Question: How do I apply for an Equitable Assessment Arrangement?

------------------------------
Rank: 1
Source: POL-01
Category: Policy
Section: Equitable Assessment Arrangements
Score: 0.7584

------------------------------
Rank: 2
Source: POL-01
Category: Policy
Section: Equitable Assessment Arrangements
Score: 0.7454

------------------------------
Rank: 3
Source: POL-01
Category: Policy
Section: Equitable Assessment Arrangements
Score: 0.7323

------------------------------
Rank: 4
Source: POL-01
Category: Policy
Section: Equitable Assessment Arrangements
Score: 0.7132

------------------------------
Rank: 5
Source: EAA-02
Category: Equitable Assessment Arrangements
Section: My teacher has directed me to apply for special consideration even though I have the Equitable Assessment Arrangement for extensions. What do I do?
Score: 0.7083

------------------------------
Rank: 6
Source: EAA-02
Category: Equitable Assessment Arrangements
Section: Do my Equitable Assessment Arrangements 

In [301]:
question = "How do I apply for an Equitable Assessment Arrangement?"

query_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(query_embedding)

# Search all 127 chunks
scores, indices = improved_index.search(query_embedding, len(chunk_df))

for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    row = chunk_df.iloc[idx]

    if row["source_id"] == "EAA-01":
        print("First EAA-01 result")
        print("Rank:", rank)
        print("Section:", row["section"])
        print("Score:", round(float(score), 4))
        print("Content:", row["content"][:500])
        break

First EAA-01 result
Rank: 8
Section: What is an EAA?
Score: 0.6784
Content: An Equitable Assessment Arrangement (EAA) is an adjustment to the standard conditions or format of an assessment for students with a disability, long-term illness and/or mental health condition. An EAA may also be available to primary carers of a close family member with a disability, long-term illness and/or mental health condition, where this means that an adjustment is needed to participate in an assessment activity. EAAs can apply to any formal assessment task described in the course guide. 


In [302]:
question = "How do I apply for an Equitable Assessment Arrangement?"

query_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(query_embedding)

scores, indices = improved_index.search(
    query_embedding,
    len(chunk_df)
)

for rank, (idx, score) in enumerate(
    zip(indices[0], scores[0]),
    start=1
):
    row = chunk_df.iloc[idx]

    if (
        row["source_id"] == "EAA-01"
        and "How to apply" in str(row["section"])
    ):
        print("EAA-01 How to apply")
        print("Rank:", rank)
        print("Section:", row["section"])
        print("Score:", round(float(score), 4))
        print("Content:", row["content"][:700])
        break

EAA-01 How to apply
Rank: 13
Section: How to apply
Score: 0.654
Content: To access an EAA, you must register with Equitable Learning Services (ELS) . It is recommended that you register at least six weeks before a major timed assessment or four weeks before your first assessment task. If you register after this time, we may not be able to offer a full range of assistance. ELS will require supporting documentation from a health practitioner (e.g. a doctor, psychologist or social worker) who is treating your condition and is registered with a recognised professional body.


In [303]:
def retrieve_context(question, search_index, top_k=5):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = search_index.search(
        query_embedding,
        top_k
    )

    retrieved_chunks = []

    for idx, score in zip(indices[0], scores[0]):

        row = chunk_df.iloc[idx]

        retrieved_chunks.append({
            "source_id": row["source_id"],
            "category": row["category"],
            "section": row["section"],
            "content": row["content"],
            "source_url": row["source_url"],
            "score": round(float(score), 4)
        })

    return retrieved_chunks

In [304]:
question = "Can I get an extension for an assignment?"

retrieved_context = retrieve_context(
    question,
    improved_index,
    top_k=1
)

In [305]:
for rank, chunk in enumerate(retrieved_context, start=1):
    print(f"\n--- Rank {rank} ---")
    print("Source:", chunk["source_id"])
    print("Section:", chunk["section"])
    print("Score:", chunk["score"])
    print("Content:", chunk["content"][:500])


--- Rank 1 ---
Source: EXT-01
Section: Assessments eligible for an extension
Score: 0.5881
Content: You can apply for an extension for assessments which have a deadline, such as assignments, essays and projects, but not for timed assessments such as tests, exams, quizzes or lab/practical assessments. If you want to apply for assessment flexibility for a timed assessment, you must apply for special consideration .


In [306]:
def build_context(retrieved_chunks):

    context_parts = []

    for rank, chunk in enumerate(retrieved_chunks, start=1):

        context_part = f"""
Source {rank}
Source ID: {chunk['source_id']}
Category: {chunk['category']}
Section: {chunk['section']}
Content: {chunk['content']}
URL: {chunk['source_url']}
"""

        context_parts.append(context_part)

    return "\n".join(context_parts)


context_text = build_context(retrieved_context)

print(context_text)


Source 1
Source ID: EXT-01
Category: Extension
Section: Assessments eligible for an extension
Content: You can apply for an extension for assessments which have a deadline, such as assignments, essays and projects, but not for timed assessments such as tests, exams, quizzes or lab/practical assessments. If you want to apply for assessment flexibility for a timed assessment, you must apply for special consideration .
URL: https://www.rmit.edu.au/students/my-course/assessment-results/special-consideration-extensions/extensions



In [307]:
def build_prompt(question, context):

    prompt = f"""
You are an assistant helping RMIT students understand assessment support.

Answer the student's question using only the information provided in the context below.

Rules:
- Do not invent information.
- If the context does not contain enough information, say that you do not have enough information.
- Answer in 2 to 3 complete sentences.
- Explain the answer using the information in the context.
- Distinguish general assessment extension information from Equitable Learning Plan (ELP) arrangements where relevant.
- Base the answer only on the retrieved RMIT information.

Student question:
{question}

Context:
{context}

Answer:
"""

    return prompt


rag_prompt = build_prompt(
    question,
    context_text
)

print(rag_prompt)


You are an assistant helping RMIT students understand assessment support.

Answer the student's question using only the information provided in the context below.

Rules:
- Do not invent information.
- If the context does not contain enough information, say that you do not have enough information.
- Answer in 2 to 3 complete sentences.
- Explain the answer using the information in the context.
- Distinguish general assessment extension information from Equitable Learning Plan (ELP) arrangements where relevant.
- Base the answer only on the retrieved RMIT information.

Student question:
Can I get an extension for an assignment?

Context:

Source 1
Source ID: EXT-01
Category: Extension
Section: Assessments eligible for an extension
Content: You can apply for an extension for assessments which have a deadline, such as assignments, essays and projects, but not for timed assessments such as tests, exams, quizzes or lab/practical assessments. If you want to apply for assessment flexibility 

In [308]:
inputs = tokenizer(
    rag_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

rag_answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("RAG answer:")
print(rag_answer)

RAG answer:
Yes


In [309]:
import transformers

print("Transformers version:", transformers.__version__)

Transformers version: 4.57.6


In [310]:
print("Chunks:", len(chunk_df))
print("Improved index vectors:", improved_index.ntotal)
print("Question:", question)
print("Prompt ready:", len(rag_prompt) > 0)

Chunks: 127
Improved index vectors: 127
Question: Can I get an extension for an assignment?
Prompt ready: True


In [311]:
import sys
import torch
import transformers

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: 3.9.6 (tags/v3.9.6:db3ff76, Jun 28 2021, 15:26:21) [MSC v.1929 64 bit (AMD64)]
PyTorch: 2.8.0+cpu
Transformers: 4.57.6
CUDA available: False


In [312]:
from transformers import AutoTokenizer

print("AutoTokenizer imported successfully")

AutoTokenizer imported successfully


In [313]:
print("Kernel is working")

Kernel is working


In [314]:
import huggingface_hub

print("Hugging Face Hub imported successfully")
print("Version:", huggingface_hub.__version__)

Hugging Face Hub imported successfully
Version: 0.36.2


In [315]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="google/flan-t5-small",
    filename="config.json"
)

print("Download successful")
print(file_path)

Download successful
C:\Users\krbha\.cache\huggingface\hub\models--google--flan-t5-small\snapshots\0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab\config.json


In [316]:
import sentencepiece as spm

print("SentencePiece imported successfully")
print("Version:", spm.__version__)

SentencePiece imported successfully
Version: 0.2.0


In [317]:
from transformers import AutoTokenizer

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("FLAN-T5 tokenizer loaded successfully.")

FLAN-T5 tokenizer loaded successfully.


In [318]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-small"
)

print("FLAN-T5 model loaded successfully.")

FLAN-T5 model loaded successfully.


In [319]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

base_model_name = "google/flan-t5-base"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)

print("FLAN-T5-base loaded successfully.")

FLAN-T5-base loaded successfully.


In [320]:
print(type(tokenizer))
print(type(model))

<class 'transformers.models.t5.tokenization_t5_fast.T5TokenizerFast'>
<class 'transformers.models.t5.modeling_t5.T5ForConditionalGeneration'>


In [321]:
test_prompt = "What is an assessment extension?"

inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
)

outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Generated answer:")
print(answer)

Generated answer:
a re-examine


In [322]:
test_prompt = "What is an assessment extension?"

inputs = base_tokenizer(
    test_prompt,
    return_tensors="pt"
)

outputs = base_model.generate(
    **inputs,
    max_new_tokens=50
)

answer = base_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("FLAN-T5-base generated answer:")
print(answer)

FLAN-T5-base generated answer:
a statutory extension


In [323]:
encoded = tokenizer(
    rag_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

print("Tokens sent to model:", encoded["input_ids"].shape[1])

decoded_input = tokenizer.decode(
    encoded["input_ids"][0],
    skip_special_tokens=True
)

print("\nActual prompt received by model:\n")
print(decoded_input)

Tokens sent to model: 270

Actual prompt received by model:

You are an assistant helping RMIT students understand assessment support. Answer the student's question using only the information provided in the context below. Rules: - Do not invent information. - If the context does not contain enough information, say that you do not have enough information. - Answer in 2 to 3 complete sentences. - Explain the answer using the information in the context. - Distinguish general assessment extension information from Equitable Learning Plan (ELP) arrangements where relevant. - Base the answer only on the retrieved RMIT information. Student question: Can I get an extension for an assignment? Context: Source 1 Source ID: EXT-01 Category: Extension Section: Assessments eligible for an extension Content: You can apply for an extension for assessments which have a deadline, such as assignments, essays and projects, but not for timed assessments such as tests, exams, quizzes or lab/practical assess